# Введение в MapReduce модель на Python


In [260]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

In [261]:
def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

Модель элемента данных

In [262]:
class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

In [263]:
input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

Функция RECORDREADER моделирует чтение элементов с диска или по сети.

In [264]:
def RECORDREADER():
  return [(u.id, u) for u in input_collection]

In [265]:
list(RECORDREADER())

[(0, User(id=0, age=55, social_contacts=20, gender='male')),
 (1, User(id=1, age=25, social_contacts=240, gender='female')),
 (2, User(id=2, age=25, social_contacts=500, gender='female')),
 (3, User(id=3, age=33, social_contacts=800, gender='female'))]

In [266]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

In [267]:
map_output = flatten(map(lambda x: MAP(*x), RECORDREADER()))
map_output = list(map_output) # materialize
map_output

[(25, User(id=1, age=25, social_contacts=240, gender='female')),
 (25, User(id=2, age=25, social_contacts=500, gender='female')),
 (33, User(id=3, age=33, social_contacts=800, gender='female'))]

In [268]:
def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

In [269]:
shuffle_output = groupbykey(map_output)
shuffle_output = list(shuffle_output)
shuffle_output

[(25,
  [User(id=1, age=25, social_contacts=240, gender='female'),
   User(id=2, age=25, social_contacts=500, gender='female')]),
 (33, [User(id=3, age=33, social_contacts=800, gender='female')])]

In [270]:
reduce_output = flatten(map(lambda x: REDUCE(*x), shuffle_output))
reduce_output = list(reduce_output)
reduce_output

[(25, 370.0), (33, 800.0)]

Все действия одним конвейером!

In [271]:
list(flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER()))))))

[(25, 370.0), (33, 800.0)]

# **MapReduce**
Выделим общую для всех пользователей часть системы в отдельную функцию высшего порядка. Это наиболее простая модель MapReduce, без учёта распределённого хранения данных.

Пользователь для решения своей задачи реализует RECORDREADER, MAP, REDUCE.

In [272]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

## Спецификация MapReduce



```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

mapreduce ((k1,v1)*) -> (k3,v3)*
groupby ((k2,v2)*) -> (k2,v2*)*
flatten (e2**) -> e2*

mapreduce .map(f).flatten.groupby(k2).map(g).flatten
```




# Примеры

## SQL

In [273]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

def RECORDREADER():
  return [(u.id, u) for u in input_collection]

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(25, 370.0), (33, 800.0)]

## Matrix-Vector multiplication

In [274]:
from typing import Iterator
import numpy as np

mat = np.ones((5,4))
vec = np.random.rand(4) # in-memory vector in all map tasks

def MAP(coordinates:(int, int), value:int):
  i, j = coordinates
  yield (i, value*vec[j])

def REDUCE(i:int, products:Iterator[NamedTuple]):
  sum = 0
  for p in products:
    sum += p
  yield (i, sum)

def RECORDREADER():
  for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
      yield ((i, j), mat[i,j])

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(0, 1.920864558434293),
 (1, 1.920864558434293),
 (2, 1.920864558434293),
 (3, 1.920864558434293),
 (4, 1.920864558434293)]

## Inverted index

In [275]:
from typing import Iterator

d1 = "it is what it is"
d2 = "what is it"
d3 = "it is a banana"
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    yield ("{}".format(docid), document)

def MAP(docId:str, body:str):
  for word in set(body.split(' ')):
    yield (word, docId)

def REDUCE(word:str, docIds:Iterator[str]):
  yield (word, sorted(docIds))

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('it', ['0', '1', '2']),
 ('is', ['0', '1', '2']),
 ('what', ['0', '1']),
 ('a', ['2']),
 ('banana', ['2'])]

## WordCount

In [276]:
from typing import Iterator

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    for (lineid, line) in enumerate(document.split('\n')):
      yield ("{}:{}".format(docid,lineid), line)

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('', 3), ('it', 9), ('is', 9), ('what', 5), ('a', 1), ('banana', 1)]

# MapReduce Distributed

Добавляется в модель фабрика RECORDREARER-ов --- INPUTFORMAT, функция распределения промежуточных результатов по партициям PARTITIONER, и функция COMBINER для частичной аггрегации промежуточных результатов до распределения по новым партициям.

In [277]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def groupbykey_distributed(map_partitions, PARTITIONER):
  global reducers
  partitions = [dict() for _ in range(reducers)]
  for map_partition in map_partitions:
    for (k2, v2) in map_partition:
      p = partitions[PARTITIONER(k2)]
      p[k2] = p.get(k2, []) + [v2]
  return [(partition_id, sorted(partition.items(), key=lambda x: x[0])) for (partition_id, partition) in enumerate(partitions)]

def PARTITIONER(obj):
  global reducers
  return hash(obj) % reducers

def MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER=PARTITIONER, COMBINER=None):
  map_partitions = map(lambda record_reader: flatten(map(lambda k1v1: MAP(*k1v1), record_reader)), INPUTFORMAT())
  if COMBINER != None:
    map_partitions = map(lambda map_partition: flatten(map(lambda k2v2: COMBINER(*k2v2), groupbykey(map_partition))), map_partitions)
  reduce_partitions = groupbykey_distributed(map_partitions, PARTITIONER) # shuffle
  reduce_outputs = map(lambda reduce_partition: (reduce_partition[0], flatten(map(lambda reduce_input_group: REDUCE(*reduce_input_group), reduce_partition[1]))), reduce_partitions)

  print("{} key-value pairs were sent over a network.".format(sum([len(vs) for (k,vs) in flatten([partition for (partition_id, partition) in reduce_partitions])])))
  return reduce_outputs

## Спецификация MapReduce Distributed


```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

e1 (k1, v1)
e2 (k2, v2)
partition1 (k2, v2)*
partition2 (k2, v2*)*

flatmap (e1->e2*, e1*) -> partition1*
groupby (partition1*) -> partition2*

mapreduce ((k1,v1)*) -> (k3,v3)*
mapreduce .flatmap(f).groupby(k2).flatmap(g)
```



## WordCount

In [278]:
from typing import Iterator
import numpy as np

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3, d1, d2, d3]

maps = 3
reducers = 2

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for (docid, document) in enumerate(split):
      for (lineid, line) in enumerate(document.split('\n')):
        yield ("{}:{}".format(docid,lineid), line)

  split_size =  int(np.ceil(len(documents)/maps))
  for i in range(0, len(documents), split_size):
    yield RECORDREADER(documents[i:i+split_size])

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

# try to set COMBINER=REDUCER and look at the number of values sent over the network
partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

56 key-value pairs were sent over a network.


[(0, [('', 6), ('is', 18), ('what', 10)]),
 (1, [('a', 2), ('banana', 2), ('it', 18)])]

## TeraSort

In [279]:
import numpy as np

input_values = np.random.rand(30)
maps = 3
reducers = 2
min_value = 0.0
max_value = 1.0

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for value in split:
        yield (value, None)

  split_size =  int(np.ceil(len(input_values)/maps))
  for i in range(0, len(input_values), split_size):
    yield RECORDREADER(input_values[i:i+split_size])

def MAP(value:int, _):
  yield (value, None)

def PARTITIONER(key):
  global reducers
  global max_value
  global min_value
  bucket_size = (max_value-min_value)/reducers
  bucket_id = 0
  while((key>(bucket_id+1)*bucket_size) and ((bucket_id+1)*bucket_size<max_value)):
    bucket_id += 1
  return bucket_id

def REDUCE(value:int, _):
  yield (None,value)

partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None, PARTITIONER=PARTITIONER)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

30 key-value pairs were sent over a network.


[(0,
  [(None, 0.03101602636271894),
   (None, 0.043461034998028225),
   (None, 0.057638418969272576),
   (None, 0.08693053799710271),
   (None, 0.151367738417008),
   (None, 0.1694997395867005),
   (None, 0.22161983019823728),
   (None, 0.23499494476532512),
   (None, 0.25730210189261815),
   (None, 0.32119518112267476),
   (None, 0.3245614663238994),
   (None, 0.3336817986350894),
   (None, 0.342402867974284),
   (None, 0.41401376870052453),
   (None, 0.415119212934116),
   (None, 0.468069460214508),
   (None, 0.49467205485714505)]),
 (1,
  [(None, 0.5280078560703323),
   (None, 0.5524943314801698),
   (None, 0.6139696447906637),
   (None, 0.6242461000311539),
   (None, 0.6696037292260091),
   (None, 0.6816771840286345),
   (None, 0.7037675071934045),
   (None, 0.7420978581928033),
   (None, 0.8152199838363503),
   (None, 0.8256410304812541),
   (None, 0.8741578731194961),
   (None, 0.8784315308434996),
   (None, 0.99282672585357)])]

# Упражнения
Упражнения взяты из Rajaraman A., Ullman J. D. Mining of massive datasets. – Cambridge University Press, 2011.


Для выполнения заданий переопределите функции RECORDREADER, MAP, REDUCE. Для модели распределённой системы может потребоваться переопределение функций PARTITION и COMBINER.

### Максимальное значение ряда

Разработайте MapReduce алгоритм, который находит максимальное число входного списка чисел.

In [280]:
from typing import NamedTuple, Iterator

class DataRecord(NamedTuple):
    source_id: int
    value: int

input_data = [
    DataRecord(source_id=1, value=57),
    DataRecord(source_id=2, value=23),
    DataRecord(source_id=3, value=99), 
    DataRecord(source_id=4, value=23)
]

def RECORDREADER():
    return [(u.source_id, u) for u in input_data]

def MAP(_, record: DataRecord):
    yield ("max", record.value)

def REDUCE(key: str, values: Iterator[int]):
    val_list = list(values)
    if not val_list:
        return
    result = max(val_list)
    yield (key, result)

def flatten(nested_iterable):
    for iterable in nested_iterable:
        for element in iterable:
            yield element

def groupbykey(iterable):
    groups = {}
    for k, v in iterable:
        if k not in groups:
            groups[k] = []
        groups[k].append(v)
    return groups.items()

map_output = list(flatten(map(lambda x: MAP(*x), RECORDREADER())))
shuffle_output = list(groupbykey(map_output))
reduce_output = list(flatten(map(lambda x: REDUCE(*x), shuffle_output)))

print("MAP output:", map_output)
print("SHUFFLE output:", shuffle_output)
print("RESULT:", reduce_output)

MAP output: [('max', 57), ('max', 23), ('max', 99), ('max', 23)]
SHUFFLE output: [('max', [57, 23, 99, 23])]
RESULT: [('max', 99)]


### Арифметическое среднее

Разработайте MapReduce алгоритм, который находит арифметическое среднее.

$$\overline{X} = \frac{1}{n}\sum_{i=0}^{n} x_i$$


In [281]:
from typing import NamedTuple, Iterator, Tuple, List

class DataRecord(NamedTuple):
    source_id: int
    value: int

input_data = [
    DataRecord(source_id=1, value=57),
    DataRecord(source_id=2, value=23),
    DataRecord(source_id=3, value=99), 
    DataRecord(source_id=4, value=23)
]

def RECORDREADER():
    return [(u.source_id, u) for u in input_data]

def MAP(key: int, record: DataRecord):
    yield ("mean", (record.value, 1))

def COMBINER(group_key: str, partial_values: Iterator[Tuple[float, int]]):
    local_sum = 0.0
    local_count = 0
    
    for val, cnt in partial_values:
        local_sum += val
        local_count += cnt
    
    yield (group_key, (local_sum, local_count))

def REDUCE(group_key: str, aggregated_values: Iterator[Tuple[float, int]]):
    global_sum = 0.0
    global_count = 0
    for partial_sum, partial_count in aggregated_values:
        global_sum += partial_sum
        global_count += partial_count
    
    average = global_sum / global_count if global_count > 0 else 0.0
    yield (group_key, average)

def flatten(nested_iterable):
    for iterable in nested_iterable:
        for element in iterable:
            yield element

def groupbykey(iterable):
    groups = {}
    for k, v in iterable:
        if k not in groups:
            groups[k] = []
        groups[k].append(v)
    return groups.items()

map_output = list(flatten(map(lambda x: MAP(*x), RECORDREADER())))
combiner_output = list(flatten(map(lambda x: COMBINER(*x), groupbykey(map_output))))
reduce_output = list(flatten(map(lambda x: REDUCE(*x), groupbykey(combiner_output))))

print(f"MAP output: {map_output}")
print(f"COMBINER output: {combiner_output}")
print(f"REDUCE output: {reduce_output}")


MAP output: [('mean', (57, 1)), ('mean', (23, 1)), ('mean', (99, 1)), ('mean', (23, 1))]
COMBINER output: [('mean', (202.0, 4))]
REDUCE output: [('mean', 50.5)]


### GroupByKey на основе сортировки

Реализуйте groupByKey на основе сортировки, проверьте его работу на примерах

In [282]:
def groupbykey_sorted(iterable):
    if not iterable:
        return []
    
    sorted_pairs = sorted(iterable, key=lambda x: x[0])
    
    grouped = []
    for (key, val) in sorted_pairs:
        if  len(grouped) == 0 or grouped[- 1][0] != key:
            grouped.append((key, [val]))
        else:
            grouped[-1][1].append(val)
    
    return grouped

In [283]:
print(groupbykey_sorted([(5,"a"), (50,"c"), (5,"b")]))
pairs1 = [
  (4, "a"), (-1, "x"), (4, "b"), (2, "k"), (-4, "y"),
  (0, "z"), (4, "m"), (3, "c"), (1, "q"), (-1, "w")
]
print(groupbykey_sorted(pairs1))

pairs2 = [
  ("apple", 1), ("banana", None), ("banana", "mmm"), ("apple", 8),
  ("banana", "beee"), ("apple", 3), ("pear", 0), ("pear", 5)
]
print(groupbykey_sorted(pairs2))

pairs3 = [
  (2, [1, 2]), (1, [9]), (2, [3]), (1, []), (3, [7, 8]), (2, [])
]
print(groupbykey_sorted(pairs3))


[(5, ['a', 'b']), (50, ['c'])]
[(-4, ['y']), (-1, ['x', 'w']), (0, ['z']), (1, ['q']), (2, ['k']), (3, ['c']), (4, ['a', 'b', 'm'])]
[('apple', [1, 8, 3]), ('banana', [None, 'mmm', 'beee']), ('pear', [0, 5])]
[(1, [[9], []]), (2, [[1, 2], [3], []]), (3, [[7, 8]])]


### Drop duplicates (set construction, unique elements, distinct)

Реализуйте распределённую операцию исключения дубликатов

In [284]:
from typing import NamedTuple, Iterator, List

class DataItem(NamedTuple):
    item_id: int
    content: str

dataset = [
    DataItem(0, "apple"),
    DataItem(1, "banana"),
    DataItem(2, "apple"),   
    DataItem(3, "cherry"),
    DataItem(4, "banana"),  
    DataItem(5, "date"),
    DataItem(6, "date")     
]

def RECORDREADER(data: List[DataItem]):
    for item in data:
        yield (item.item_id, item.content)

def MAP(key: int, value: str):
    yield (value, 1)

def REDUCE(key: str, values: Iterator[int]):
    yield key

map_phase = list(flatten(map(lambda x: MAP(*x), RECORDREADER(dataset))))
print(f"После MAP: {map_phase}")

shuffle_phase = list(groupbykey(map_phase))
print(f"После SHUFFLE: {shuffle_phase}")

reduce_phase = list(flatten(map(lambda x: REDUCE(*x), shuffle_phase)))
print(f"Уникальные элементы: {sorted(reduce_phase)}")

После MAP: [('apple', 1), ('banana', 1), ('apple', 1), ('cherry', 1), ('banana', 1), ('date', 1), ('date', 1)]
После SHUFFLE: [('apple', [1, 1]), ('banana', [1, 1]), ('cherry', [1]), ('date', [1, 1])]
Уникальные элементы: ['apple', 'banana', 'cherry', 'date']


#Операторы реляционной алгебры
### Selection (Выборка)

**The Map Function**: Для  каждого кортежа $t \in R$ вычисляется истинность предиката $C$. В случае истины создаётся пара ключ-значение $(t, t)$. В паре ключ и значение одинаковы, равны $t$.

**The Reduce Function:** Роль функции Reduce выполняет функция идентичности, которая возвращает то же значение, что получила на вход.



In [285]:
R = [
    ("Ivan", 5),
    ("Masha", 31),
    ("Sergay", 60),
    ("Olga", 19),
    ("Dima", 81),
    ("Katya", 19),
    ("Pepe", 81),
]

def C(t):
    return t[1] < 50

def RECORDREADER():
    for t in R:
        yield (t, t)

def MAP(_, t):
    if C(t):
        yield (t, t)

def REDUCE(t, values: Iterator):
    yield (t, t)
    
def MapReduce(RECORDREADER, MAP, REDUCE):
    return flatten(
        map(lambda kv: REDUCE(*kv),
            groupbykey(
                flatten(map(lambda x: MAP(*x), RECORDREADER()))
            )
        )
    )

selected = [t for (t, _) in MapReduce(RECORDREADER, MAP, REDUCE)]
print("Исходное отношение:")
print(R)

print("Результат:")
print(selected)

Исходное отношение:
[('Ivan', 5), ('Masha', 31), ('Sergay', 60), ('Olga', 19), ('Dima', 81), ('Katya', 19), ('Pepe', 81)]
Результат:
[('Ivan', 5), ('Masha', 31), ('Olga', 19), ('Katya', 19)]


### Projection (Проекция)

Проекция на множество атрибутов $S$.

**The Map Function:** Для каждого кортежа $t \in R$ создайте кортеж $t′$, исключая  из $t$ те значения, атрибуты которых не принадлежат  $S$. Верните пару $(t′, t′)$.

**The Reduce Function:** Для каждого ключа $t′$, созданного любой Map задачей, вы получаете одну или несколько пар $(t′, t′)$. Reduce функция преобразует $(t′, [t′, t′, . . . , t′])$ в $(t′, t′)$, так, что для ключа $t′$ возвращается одна пара  $(t′, t′)$.

In [286]:
S_idx = [1]  

def project(t, idxs):
    return tuple(t[i] for i in idxs)

def RECORDREADER():
    for t in R:
        yield (t, t)

def MAP(_, t):
    t2 = project(t, S_idx)
    yield (t2, t2)

def REDUCE(t, values: Iterator):
    yield (t, t)

projection = [t2 for (t2, _) in MapReduce(RECORDREADER, MAP, REDUCE)]
print(projection)

[(5,), (31,), (60,), (19,), (81,)]


### Union (Объединение)

**The Map Function:** Превратите каждый входной кортеж $t$ в пару ключ-значение $(t, t)$.

**The Reduce Function:** С каждым ключом $t$ будет ассоциировано одно или два значений. В обоих случаях создайте $(t, t)$ в качестве выходного значения.

In [287]:
R = [
    ("Ivan", 25),
    ("Olga", 31),
]

S = [
    ("Olga", 31),   
    ("Petr", 40),
]

def RECORDREADER():
    for t in R:
        yield ("R", t)
    for t in S:
        yield ("S", t)

def MAP(_, t):
    yield (t, t)

def REDUCE(t, values: Iterator):
    yield (t, t)

union = [t for (t, _) in MapReduce(RECORDREADER, MAP, REDUCE)]
sorted(union)

[('Ivan', 25), ('Olga', 31), ('Petr', 40)]

### Intersection (Пересечение)

**The Map Function:** Превратите каждый кортеж $t$ в пары ключ-значение $(t, t)$.

**The Reduce Function:** Если для ключа $t$ есть список из двух элементов $[t, t]$ $-$ создайте пару $(t, t)$. Иначе, ничего не создавайте.

In [288]:
def RECORDREADER():
    for t in R:
        yield ("R", t)
    for t in S:
        yield ("S", t)

def MAP(rel, t):
    yield (t, rel)

def REDUCE(t, rels):
    sources = set(rels)
    if "R" in sources and "S" in sources:
        yield (t, list(sources)) 

intersect = [t for (t, _) in MapReduce(RECORDREADER, MAP, REDUCE)]
print(intersect)

[('Olga', 31)]


### Difference (Разница)

**The Map Function:** Для кортежа $t \in R$, создайте пару $(t, R)$, и для кортежа $t \in S$, создайте пару $(t, S)$. Задумка заключается в том, чтобы значение пары было именем отношения $R$ or $S$, которому принадлежит кортеж (а лучше, единичный бит, по которому можно два отношения различить $R$ or $S$), а не весь набор атрибутов отношения.

**The Reduce Function:** Для каждого ключа $t$, если соответствующее значение является списком $[R]$, создайте пару $(t, t)$. В иных случаях не предпринимайте действий.

In [289]:
def RECORDREADER():
    for t in R:
        yield ("R", t)
    for t in S:
        yield ("S", t)

def MAP(rel, t):
    yield (t, rel)

def REDUCE(t, rels):
    sources = set(rels)
    if "S" not in sources:
        yield (t, t)

diff = [t for (t, _) in MapReduce(RECORDREADER, MAP, REDUCE)]
print(diff)

[('Ivan', 25)]


### Natural Join

**The Map Function:** Для каждого кортежа $(a, b)$ отношения $R$, создайте пару $(b,(R, a))$. Для каждого кортежа $(b, c)$ отношения $S$, создайте пару $(b,(S, c))$.

**The Reduce Function:** Каждый ключ $b$ будет асоциирован со списком пар, которые принимают форму либо $(R, a)$, либо $(S, c)$. Создайте все пары, одни, состоящие из  первого компонента $R$, а другие, из первого компонента $S$, то есть $(R, a)$ и $(S, c)$. На выходе вы получаете последовательность пар ключ-значение из списков ключей и значений. Ключ не нужен. Каждое значение, это тройка $(a, b, c)$ такая, что $(R, a)$ и $(S, c)$ это принадлежат входному списку значений.

In [290]:
from typing import Iterator, Tuple

R = [
    ("Иван", "IT"),
    ("Ольга", "IT"),
    ("Петр", "HR"),
    ("Анна", "Бухгалтерия")
]
S = [
    ("IT", "Москва"),
    ("HR", "Петербург"),
    ("Маркетинг", "Казань") 
]

def RECORDREADER():
    for a, b in R:
        yield ("R", (a, b))    
    for b, c in S:
        yield ("S", (b, c))   

def MAP(rel, t):
    if rel == "R":
        a, b = t
        yield (b, ("R", a))    
    else:
        b, c = t
        yield (b, ("S", c))    

def REDUCE(b, values: Iterator[Tuple[str, any]]):
    r_values = []  
    s_values = []  
    
    for tag, value in values:
        if tag == "R":
            r_values.append(value)  
        else:
            s_values.append(value)  
    
    for a in r_values:
        for c in s_values:
            yield (None, (a, b, c))  

result = [row for (_, row) in MapReduce(RECORDREADER, MAP, REDUCE)]

print(result)

[('Иван', 'IT', 'Москва'), ('Ольга', 'IT', 'Москва'), ('Петр', 'HR', 'Петербург')]


### Grouping and Aggregation (Группировка и аггрегация)

**The Map Function:** Для каждого кортежа $(a, b, c$) создайте пару $(a, b)$.

**The Reduce Function:** Ключ представляет ту или иную группу. Примение аггрегирующую операцию $\theta$ к списку значений $[b1, b2, . . . , bn]$ ассоциированных с ключом $a$. Возвращайте в выходной поток $(a, x)$, где $x$ результат применения  $\theta$ к списку. Например, если $\theta$ это $SUM$, тогда $x = b1 + b2 + · · · + bn$, а если $\theta$ is $MAX$, тогда $x$ это максимальное из значений $b1, b2, . . . , bn$.

In [291]:
R = [
    ("Хлеб", 50, "Касса_1"),    
    ("Молоко", 80, "Касса_2"),  
    ("Хлеб", 50, "Касса_3"),    
    ("Яблоки", 120, "Касса_1"), 
    ("Молоко", 80, "Касса_1"),  
    ("Хлеб", 50, "Касса_2"),    
]

def RECORDREADER():
    for t in R:
        yield (None, t)

def MAP(_, t):
    a, b, c = t
    yield (a, b)

def REDUCE(a, bs):
    s = 0
    for b in bs:
        s += b
    yield (a, s)

grouped = list(MapReduce(RECORDREADER, MAP, REDUCE))
print(grouped)

[('Хлеб', 150), ('Молоко', 160), ('Яблоки', 120)]


#

### Matrix-Vector multiplication

Случай, когда вектор не помещается в памяти Map задачи


In [292]:
from typing import Iterator

A = [
    (0, 0, 1), (0, 1, 2), (0, 2, 3),
    (1, 0, 4), (1, 1, 5), (1, 2, 6),
]

x = [
    (0, 1),
    (1, 2),
    (2, 3),
]

def RECORDREADER():
    for i, j, a in A:
        yield ("A", (i, j, a))
    for j, xj in x:
        yield ("x", (j, xj))

def MAP1(src, rec):
    if src == "A":
        i, j, a = rec
        yield (j, ("A", i, a))
    else:
        j, xj = rec
        yield (j, ("x", xj))

def REDUCE1(j, values: Iterator[tuple]):
    xj = None
    a_entries = []

    for tag, *rest in values:
        if tag == "x":
            xj = rest[0]
        else:
            i, a = rest
            a_entries.append((i, a))

    if xj is None:
        return

    for i, a in a_entries:
        yield (i, a * xj)
        
def MAP2(key, partial):
    yield (key, partial)

def REDUCE2(key, values: Iterator[int]):
    total_sum = sum(values)
    return [(key, total_sum)]

res_map1 = list(flatten(MAP1(*x) for x in RECORDREADER()))
res_shuffle1 = list(groupbykey(res_map1))
res_job1 = list(flatten(REDUCE1(j, iter(vs)) for j, vs in res_shuffle1))

print("Промежуточные результаты:")
print("MAP:", res_map1)
print("SHUFFLE:", res_shuffle1)
print("RESULT:", res_job1)


res_map2 = list(flatten(MAP2(*p) for p in res_job1))
res_shuffle2 = list(groupbykey(res_map2))
res = list(flatten(REDUCE2(i, iter(vs)) for i, vs in res_shuffle2))

print("Финальные:")
print("MAP:", res_map2)
print("SHUFFLE:", res_shuffle2)
print("FINAL:", res)

Промежуточные результаты:
MAP: [(0, ('A', 0, 1)), (1, ('A', 0, 2)), (2, ('A', 0, 3)), (0, ('A', 1, 4)), (1, ('A', 1, 5)), (2, ('A', 1, 6)), (0, ('x', 1)), (1, ('x', 2)), (2, ('x', 3))]
SHUFFLE: [(0, [('A', 0, 1), ('A', 1, 4), ('x', 1)]), (1, [('A', 0, 2), ('A', 1, 5), ('x', 2)]), (2, [('A', 0, 3), ('A', 1, 6), ('x', 3)])]
RESULT: [(0, 1), (1, 4), (0, 4), (1, 10), (0, 9), (1, 18)]
Финальные:
MAP: [(0, 1), (1, 4), (0, 4), (1, 10), (0, 9), (1, 18)]
SHUFFLE: [(0, [1, 4, 9]), (1, [4, 10, 18])]
FINAL: [(0, 14), (1, 32)]


## Matrix multiplication (Перемножение матриц)

Если у нас есть матрица $M$ с элементами $m_{ij}$ в строке $i$ и столбце $j$, и матрица $N$ с элементами $n_{jk}$ в строке $j$ и столбце $k$, тогда их произведение $P = MN$ есть матрица $P$ с элементами $p_{ik}$ в строке $i$ и столбце $k$, где

$$p_{ik} =\sum_{j} m_{ij}n_{jk}$$

Необходимым требованием является одинаковое количество столбцов в $M$ и строк в $N$, чтобы операция суммирования по  $j$ была осмысленной. Мы можем размышлять о матрице, как об отношении с тремя атрибутами: номер строки, номер столбца, само значение. Таким образом матрица $M$ предстваляется как отношение $ M(I, J, V )$, с кортежами $(i, j, m_{ij})$, и, аналогично, матрица $N$ представляется как отношение $N(J, K, W)$, с кортежами $(j, k, n_{jk})$. Так как большие матрицы как правило разреженные (большинство значений равно 0), и так как мы можем нулевыми значениями пренебречь (не хранить), такое реляционное представление достаточно эффективно для больших матриц. Однако, возможно, что координаты $i$, $j$, и $k$ неявно закодированы в смещение позиции элемента относительно начала файла, вместо явного хранения. Тогда, функция Map (или Reader) должна быть разработана таким образом, чтобы реконструировать компоненты $I$, $J$, и $K$ кортежей из смещения.

Произведение $MN$ это фактически join, за которым следуют группировка по ключу и аггрегация. Таким образом join отношений $M(I, J, V )$ и $N(J, K, W)$, имеющих общим только атрибут $J$, создаст кортежи $(i, j, k, v, w)$ из каждого кортежа $(i, j, v) \in M$ и кортежа $(j, k, w) \in N$. Такой 5 компонентный кортеж представляет пару элементов матрицы $(m_{ij} , n_{jk})$. Что нам хотелось бы получить на самом деле, это произведение этих элементов, то есть, 4 компонентный кортеж$(i, j, k, v \times w)$, так как он представляет произведение $m_{ij}n_{jk}$. Мы представляем отношение как результат одной MapReduce операции, в которой мы можем произвести группировку и аггрегацию, с $I$ и $K$  атрибутами, по которым идёт группировка, и суммой  $V \times W$.





In [293]:
# MapReduce model
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

Реализуйте перемножение матриц с использованием модельного кода MapReduce для одной машины в случае, когда одна матрица хранится в памяти, а другая генерируется RECORDREADER-ом.

In [294]:
import numpy as np
I = 2
J = 3
K = 4*10
small_mat = np.random.rand(I,J) 
big_mat = np.random.rand(J,K)

def RECORDREADER():
  for j in range(big_mat.shape[0]):
    for k in range(big_mat.shape[1]):
      yield ((j,k), big_mat[j,k])

def MAP(k1, v1):
  (j, k) = k1
  w = v1
  for i in range(I):
        yield ((i, k), (j, small_mat[i, j], w))

def REDUCE(key, values):
  (i, k) = key
  partial_sum = 0.0
  for j, small_val, big_val in values:
      partial_sum += small_val * big_val
  yield (key, partial_sum)

Проверьте своё решение

In [295]:
# CHECK THE SOLUTION
reference_solution = np.matmul(small_mat, big_mat)
solution = MapReduce(RECORDREADER, MAP, REDUCE)

def asmatrix(reduce_output):
  reduce_output = list(reduce_output)
  I = max(i for ((i,k), vw) in reduce_output)+1
  K = max(k for ((i,k), vw) in reduce_output)+1
  mat = np.empty(shape=(I,K))
  for ((i,k), vw) in reduce_output:
    mat[i,k] = vw
  return mat

np.allclose(reference_solution, asmatrix(solution)) # should return true

True

In [296]:
reduce_output = list(MapReduce(RECORDREADER, MAP, REDUCE))
max(i for ((i,k), vw) in reduce_output)

1

Реализуйте перемножение матриц  с использованием модельного кода MapReduce для одной машины в случае, когда обе матрицы генерируются в RECORDREADER. Например, сначала одна, а потом другая.

In [297]:
import numpy as np

I = 2  
J = 3  
K = 4*10  

A = np.random.rand(I, J)
B = np.random.rand(J, K)

def RECORDREADER():
    rows_a, cols_a = A.shape
    for i in range(rows_a):
        for j in range(cols_a):
            yield ("A", (i, j, A[i, j]))
    
    rows_b, cols_b = B.shape
    for j in range(rows_b):
        for k in range(cols_b):
            yield ("B", (j, k, B[j, k]))
            
def MAP1(src, rec):
    if src == "A":
        i, j, a = rec
        yield (j, ("A", i, a))
    else: 
        j, k, b = rec
        yield (j, ("B", k, b))

def REDUCE1(j, values: Iterator[tuple]):
    a_entries = []
    b_entries = []
    for tag, *rest in values:
        if tag == "A":
            a_entries.append(rest)
        else:
            b_entries.append(rest)
    
    for i, a in a_entries:
        for k, b in b_entries:
            yield ((i, k), a * b)

def MAP2(key, partial):
    yield (key, partial)

def REDUCE2(key, values: Iterator[int]):
    i, k = key
    total_sum = sum(values)
    yield (i, k, total_sum)

res_map1 = list(flatten(MAP1(*x) for x in RECORDREADER()))
res_shuffle1 = list(groupbykey(res_map1))
res_job1 = list(flatten(REDUCE1(j, iter(vs)) for j, vs in res_shuffle1))

res_map2 = list(flatten(MAP2(*p) for p in res_job1))
res_shuffle2 = list(groupbykey(res_map2))
res_final = list(flatten(REDUCE2(i_k, iter(vs)) for i_k, vs in res_shuffle2))

mr_matrix = np.zeros((I, K))

for i, k, val in res_final:
    mr_matrix[i, k] = val
    
expected = np.dot(A, B)

print("Результат (numpy):\n", expected)
print("Результат (MapReduce):\n", mr_matrix)
print("Совпадает:", np.allclose(expected, mr_matrix))

Результат (numpy):
 [[1.11414779 1.19529107 0.44057109 0.52360269 1.29623502 0.46638691
  0.92816685 0.41920531 0.41324258 0.70356651 0.77408505 1.28683108
  0.64139386 1.09478329 1.07853618 0.76552193 0.54223286 0.81045886
  0.90416287 0.93633857 0.63642555 0.87469551 0.86642549 0.42723921
  1.12166348 1.12376983 0.66724881 0.5616408  0.41300802 0.95245843
  1.21272446 0.69388734 0.20551586 0.86549406 0.72446502 0.90408332
  0.67885723 0.68510552 1.34167072 0.41690005]
 [0.46183867 0.30923958 0.22284784 0.40076314 0.26498114 0.25860079
  0.21420767 0.27261749 0.24114069 0.28235873 0.39329709 0.49727395
  0.22936722 0.3586681  0.43492831 0.05878142 0.28350453 0.27594204
  0.09423167 0.21390887 0.29257246 0.36691003 0.19075279 0.10166744
  0.25398365 0.45920678 0.11088491 0.40015066 0.38770937 0.1842237
  0.50848141 0.31398922 0.29671046 0.19211722 0.13859957 0.31289578
  0.1544564  0.27654032 0.44079967 0.19629866]]
Результат (MapReduce):
 [[1.11414779 1.19529107 0.44057109 0.52360269 

Реализуйте перемножение матриц с использованием модельного кода MapReduce Distributed, когда каждая матрица генерируется в своём RECORDREADER.

In [298]:
import numpy as np
from itertools import chain

I = 2   
J = 3  
K = 4*10  

A = np.random.rand(I, J)
B = np.random.rand(J, K)

def RECORDREADER_A():
    rows, cols = A.shape
    for i in range(rows):
        for j in range(cols):
            yield ("A", (i, j, A[i, j]))

def RECORDREADER_B():
    rows, cols = B.shape
    for j in range(rows):
        for k in range(cols):
            yield ("B", (j, k, B[j, k]))

def MAP1(src, rec):
    if src == "A":
        i, j, a = rec
        yield (j, ("A", i, a))
    else: 
        j, k, b = rec
        yield (j, ("B", k, b))

def REDUCE1(j, values: Iterator[tuple]):
    a_entries = []  
    b_entries = [] 
    
    for tag, *rest in values:
        if tag == "A":
            i, a = rest
            a_entries.append((i, a))
        else:  
            k, b = rest
            b_entries.append((k, b))
    
    for i, a in a_entries:
        for k, b in b_entries:
            partial_product = a * b
            yield ((i, k), partial_product)

def MAP2(key, partial):
    yield (key, partial)

def REDUCE2(key, values: Iterator[float]):
    i, k = key
    total_sum = sum(values)
    yield (i, k, total_sum)

all_records = chain(RECORDREADER_A(), RECORDREADER_B())
res_map1 = list(flatten(MAP1(*x) for x in all_records))
res_shuffle1 = groupbykey(res_map1)
res_job1 = list(flatten(REDUCE1(j, iter(vs)) for j, vs in res_shuffle1))

res_map2 = list(flatten(MAP2(*p) for p in res_job1))
res_shuffle2 = groupbykey(res_map2)
res_final_list = list(flatten(REDUCE2(key, iter(vs)) for key, vs in res_shuffle2))

mr_matrix = np.zeros((I, K))
for i, k, val in res_final_list:
    mr_matrix[i, k] = val

expected = np.dot(A, B)

print("Результат (NumPy):", expected)
print("Результат (MapReduce Distributed):", mr_matrix)
print("Совпадает:", np.allclose(expected, mr_matrix))

Результат (NumPy): [[0.62857363 0.63604659 0.16468577 0.85327873 0.67000648 0.22041393
  0.54192485 0.54830145 0.56968616 0.44295206 0.39409659 0.50024074
  0.71412154 0.59828883 0.93545362 0.51334474 0.31881349 0.70062818
  0.75846339 0.7684402  0.68646261 0.10983097 0.67847335 0.57235793
  0.53055688 0.58673557 0.33529789 0.73608595 0.64759227 0.85186866
  0.4636541  0.73944175 0.6431186  0.47960892 0.52296451 0.48628603
  0.53886942 0.87513787 0.77154854 0.49374885]
 [0.44978476 0.77205032 0.18521385 0.79858834 0.75756306 0.3225892
  0.42085956 0.63299338 0.52972077 0.31381138 0.31243148 0.57564281
  0.89717713 0.68032262 0.92681607 0.48189505 0.3885824  0.88805637
  0.75529139 0.92328593 0.83609725 0.12630971 0.73230764 0.73005882
  0.37766122 0.52627673 0.25149663 0.62517339 0.7294107  0.85947847
  0.35784667 0.85926239 0.68899202 0.57638852 0.50459956 0.41915007
  0.50832203 1.01100688 0.93821727 0.32585806]]
Результат (MapReduce Distributed): [[0.62857363 0.63604659 0.16468577 0

Обобщите предыдущее решение на случай, когда каждая матрица генерируется несколькими RECORDREADER-ами, и проверьте его работоспособность. Будет ли работать решение, если RECORDREADER-ы будут генерировать случайное подмножество элементов матрицы?

In [299]:
import numpy as np
from itertools import groupby, chain
from typing import Iterator, List, Tuple

I = 2  
J = 3  
K = 4*10

A = np.random.rand(I, J)
B = np.random.rand(J, K)

def RECORDREADERS(matrix, matrix_name, num_readers=3):
    rows, cols = matrix.shape
    readers = []
    
    row_chunks = np.array_split(range(rows), num_readers)
    
    for chunk_idx, row_indices in enumerate(row_chunks):
        def make_reader(rows_to_read, name, mat):
            def reader():
                for i in rows_to_read:
                    for j in range(cols):
                        yield (name, (i, j, mat[i, j]))
            return reader
        
        readers.append(make_reader(row_indices, matrix_name, matrix))
    
    return readers

readers_A = RECORDREADERS(A, "A", num_readers=3)
readers_B = RECORDREADERS(B, "B", num_readers=2)

print(f"Создано {len(readers_A)} читателей для матрицы A")
print(f"Создано {len(readers_B)} читателей для матрицы B\n")

def MAP1(src, rec):
    if src == "A":
        i, j, a = rec
        yield (j, ("A", i, a))
    else:  
        j, k, b = rec
        yield (j, ("B", k, b))

def REDUCE1(j, values: Iterator[tuple]):
    a_entries = []
    b_entries = []
    for tag, *rest in values:
        if tag == "A":
            a_entries.append(rest)
        else:
            b_entries.append(rest)
    
    for i, a in a_entries:
        for k, b in b_entries:
            yield ((i, k), a * b)

def MAP2(key, partial):
    yield (key, partial)

def REDUCE2(key, values: Iterator[float]):
    i, k = key
    total_sum = sum(values)
    yield (i, k, total_sum)

all_records = chain(
    *(reader() for reader in readers_A),  
    *(reader() for reader in readers_B)  
)

res_map1 = list(flatten(MAP1(*x) for x in all_records))
res_shuffle1 = groupbykey(res_map1)
res_job1 = list(flatten(REDUCE1(j, iter(vs)) for j, vs in res_shuffle1))

res_map2 = list(flatten(MAP2(*p) for p in res_job1))
res_shuffle2 = groupbykey(res_map2)
res_final_list = list(flatten(REDUCE2(key, iter(vs)) for key, vs in res_shuffle2))

res_final_sorted = sorted(res_final_list, key=lambda x: (x[0], x[1]))

mr_matrix = np.zeros((I, K))
for i, k, val in res_final_sorted:
    mr_matrix[i, k] = val

expected = np.dot(A, B)

print("Результат (NumPy):", expected)
print("Результат:", mr_matrix)
print("Совпадает:", np.allclose(expected, mr_matrix))

Создано 3 читателей для матрицы A
Создано 2 читателей для матрицы B

Результат (NumPy): [[0.10475104 0.68528803 0.43953327 1.08011144 0.6860349  0.79139354
  0.55077088 0.94617044 0.50484537 0.73721712 0.90857964 1.19418831
  0.61855597 0.63419559 0.48792799 1.19110651 0.50638049 0.79894225
  0.41379785 0.52225529 0.87075682 0.89237393 0.1769938  1.12406324
  0.91399684 0.94159458 0.30451004 0.74150998 1.03040286 1.09222266
  1.09949957 0.95125808 1.17038604 0.65172014 1.10515209 0.97022464
  0.25778121 0.53145002 0.20776123 0.96649379]
 [0.33060913 0.41690529 0.19027674 0.92917031 0.82071996 0.81058824
  0.261027   0.27069321 0.69649227 0.85795661 0.58002272 0.51613904
  0.44297594 0.84852743 0.83914172 0.5061482  0.17876663 0.76893052
  0.21253638 0.84325519 0.36398698 0.38491659 0.09161454 0.95903064
  0.61304027 0.30335962 0.33941908 0.61987371 0.35696996 0.87651905
  1.03175543 0.85886653 1.04464328 0.59550292 0.95530387 0.6108825
  0.14517058 0.45049272 0.20639955 0.89031075]]
Ре

Случайные подмножества

In [300]:
def RECORDREADER_subset(matrix, matrix_name, num_readers=3, coverage=0.8):
    
    rows, cols = matrix.shape
    total_elements = rows * cols
    subset_size = int(total_elements * coverage)
    
    readers = []
    
    for _ in range(num_readers):
        def make_random_reader(mat, name, size, r, c):
            def reader():
                indices = np.random.choice(r * c, size=size, replace=False)
                for idx in indices:
                    i = idx // c
                    j = idx % c
                    yield (name, (i, j, mat[i, j]))
            return reader
        readers.append(make_random_reader(matrix, matrix_name, subset_size, rows, cols))
    return readers

random_readers_A = RECORDREADER_subset(A, "A", num_readers=4, coverage=0.7)
random_readers_B = RECORDREADER_subset(B, "B", num_readers=3, coverage=0.6)

all_random_records = chain(
    *(reader() for reader in random_readers_A),
    *(reader() for reader in random_readers_B)
)

res_map1_rand = list(flatten(MAP1(*x) for x in all_random_records))
res_shuffle1_rand = groupbykey(res_map1_rand)
res_job1_rand = list(flatten(REDUCE1(j, iter(vs)) for j, vs in res_shuffle1_rand))

res_map2_rand = list(flatten(MAP2(*p) for p in res_job1_rand))
res_shuffle2_rand = groupbykey(res_map2_rand)
res_final_rand = list(flatten(REDUCE2(key, iter(vs)) for key, vs in res_shuffle2_rand))

mr_matrix_rand = np.zeros((I, K))
for i, k, val in res_final_rand:
    mr_matrix_rand[i, k] = val

print("Результат:", mr_matrix_rand)


Результат: [[ 0.41435025  3.3573642   1.72502196  0.95669766  3.59759084  4.31776725
   0.16531142  3.23066157  3.05446329  5.38656047  3.55277118  8.30893411
   3.89005359  1.6429495   3.55542169 10.8501739   1.76160179  8.55961434
   2.62197188  2.55614682  6.26939988  5.59308911  0.65726268  4.21860634
   9.15991123  3.55943238  1.29378933  7.56144139  6.88558439  6.58157767
   7.38790643  3.45604563  7.44122993  0.52472909  7.5761918   9.78006571
   1.08488015  3.9984098   1.25733228  7.65805755]
 [ 3.92197235  2.86227132  1.38981146  8.80692697  6.31244686  3.2256686
   1.63948298  0.98211098  5.22218104  9.15034958  3.42539399  3.22977845
   3.10376346  3.23146246  3.47948074  3.50197466  0.21590865  5.81042677
   1.39682243  9.6784196   1.44936435  3.29208803  0.50320285  6.63968015
   2.8588233   1.19900861  3.45839973  4.63740127  2.62380329  6.18444373
   7.37713233  5.86671144 10.7893775   5.52945313  6.78351506  4.45133036
   1.1198572   3.1402769   1.49792284  3.83234237]]